In [2]:
import sqlite3
import pandas as pd

DB_PATH = "wdi.sqlite"

country_df = pd.read_csv("Country.csv")
indicators_df = pd.read_csv("Indicators.csv")

conn = sqlite3.connect(DB_PATH)

country_df.to_sql("Country", conn, if_exists="replace", index=False)
indicators_df.to_sql("Indicators", conn, if_exists="replace", index=False)

print("Database built successfully with tables: Country, Indicators")

def run_query(query, params=None):
    return pd.read_sql_query(query, conn, params=params)

Database built successfully with tables: Country, Indicators


In [3]:
highest_gdp_2014 = run_query("""
    SELECT CountryName, Value
    FROM Indicators
    WHERE IndicatorName = 'GDP per capita (current US$)'
      AND Year = 2014
    ORDER BY Value DESC
    LIMIT 1;
""")
print(highest_gdp_2014)

  CountryName          Value
0  Luxembourg  116664.262488


In [4]:
ranked_gdp_2014 = run_query("""
    SELECT CountryName, Value
    FROM Indicators
    WHERE IndicatorName = 'GDP per capita (current US$)'
      AND Year = 2014
    ORDER BY Value DESC;
""")
print(ranked_gdp_2014.head(10))

        CountryName          Value
0        Luxembourg  116664.262488
1            Norway   97307.421592
2             Qatar   96732.402545
3  Macao SAR, China   96038.050724
4       Switzerland   85594.326601
5         Australia   61925.496062
6           Denmark   60707.249365
7            Sweden   58938.772748
8         Singapore   56284.578405
9     United States   54629.495168


In [5]:
avg_gdp_per_country = run_query("""
    SELECT CountryName, AVG(Value) AS avg_gdp_per_capita
    FROM Indicators
    WHERE IndicatorName = 'GDP per capita (current US$)'
    GROUP BY CountryName
    ORDER BY avg_gdp_per_capita DESC;
""")
print(avg_gdp_per_country.head(10))

avg_gdp_selected = run_query("""
    SELECT CountryName, AVG(Value) AS avg_gdp_per_capita
    FROM Indicators
    WHERE IndicatorName = 'GDP per capita (current US$)'
      AND CountryName IN ('Brazil', 'China', 'India')
    GROUP BY CountryName
    ORDER BY avg_gdp_per_capita DESC;
""")
print(avg_gdp_selected)

            CountryName  avg_gdp_per_capita
0                Monaco        77367.086708
1         Liechtenstein        58386.073030
2       Channel Islands        50912.094992
3        Cayman Islands        47147.710137
4            San Marino        42151.387998
5        Faeroe Islands        38110.698230
6            Luxembourg        37836.168711
7           Switzerland        35770.324923
8  United Arab Emirates        33328.605338
9                 Qatar        32297.668737
  CountryName  avg_gdp_per_capita
0      Brazil         3347.485596
1       China         1111.463452
2       India          439.269836


In [6]:
max_year = run_query("SELECT MAX(Year) AS max_year FROM Indicators;")["max_year"][0]
start_year = max_year - 9

measures_last_10y = run_query("""
    SELECT COUNT(*) AS n_measures
    FROM Indicators
    WHERE Year BETWEEN ? AND ?;
""", params=(start_year, max_year))
print(f"Between {start_year} and {max_year}:")
print(measures_last_10y)

measures_per_year = run_query("""
    SELECT Year, COUNT(*) AS n_measures
    FROM Indicators
    WHERE Year BETWEEN ? AND ?
    GROUP BY Year
    ORDER BY Year;
""", params=(start_year, max_year))
print(measures_per_year)

Between 2006 and 2015:
   n_measures
0           0
Empty DataFrame
Columns: [Year, n_measures]
Index: []


In [ ]:
measures_per_country = run_query("""
    SELECT CountryName, COUNT(*) AS n_measures
    FROM Indicators
    GROUP BY CountryName
    ORDER BY n_measures ASC;
""")
print(measures_per_country.head(10))

angola_measures = run_query("""
    SELECT CountryName, COUNT(*) AS n_measures
    FROM Indicators
    WHERE CountryName = 'Angola'
    GROUP BY CountryName;
""")
print(angola_measures)

angola_by_year = run_query("""
    SELECT Year, COUNT(*) AS n_measures
    FROM Indicators
    WHERE CountryName = 'Angola'
    GROUP BY Year
    ORDER BY Year;
""")
print(angola_by_year)

                 CountryName  n_measures
0   St. Martin (French part)         572
1  Sint Maarten (Dutch part)         681
2                    Curacao        1564
3                Isle of Man        1764
4   Northern Mariana Islands        1942
5            Channel Islands        2739
6             American Samoa        2802
7   Turks and Caicos Islands        2840
8                South Sudan        3667
9                     Monaco        3881
  CountryName  n_measures
0      Angola       21158
    Year  n_measures
0   1960          70
1   1961          81
2   1962          88
3   1963          81
4   1964          74
5   1965          89
6   1966          81
7   1967          87
8   1968          81
9   1969          93
10  1970         113
11  1971         145
12  1972         151
13  1973         133
14  1974         131
15  1975         114
16  1976         117
17  1977         121
18  1978         127
19  1979         128
20  1980         145
21  1981         168
22  1982      

In [8]:
tables = run_query("SELECT name FROM sqlite_master WHERE type='table';")
print(tables)

         name
0     Country
1  Indicators


In [9]:
brazil_gdp_2014_join = run_query("""
    SELECT
        i.CountryName,
        i.CountryCode,
        i.IndicatorName,
        i.IndicatorCode,
        i.Year,
        i.Value,
        c.Region,
        c.IncomeGroup
    FROM Indicators AS i
    JOIN Country AS c
        ON i.CountryCode = c.CountryCode
    WHERE i.CountryName = 'Brazil'
      AND i.Year = 2014
      AND i.IndicatorName LIKE '%GDP%'
    ORDER BY i.IndicatorName;
""")
print(brazil_gdp_2014_join)

   CountryName CountryCode                                      IndicatorName  \
0       Brazil         BRA                Agriculture, value added (% of GDP)   
1       Brazil         BRA                             Broad money (% of GDP)   
2       Brazil         BRA         Claims on central government, etc. (% GDP)   
3       Brazil         BRA  Claims on other sectors of the domestic econom...   
4       Brazil         BRA                 Current account balance (% of GDP)   
5       Brazil         BRA  Discrepancy in expenditure estimate of GDP (cu...   
6       Brazil         BRA  Domestic credit provided by financial sector (...   
7       Brazil         BRA       Domestic credit to private sector (% of GDP)   
8       Brazil         BRA  Domestic credit to private sector by banks (% ...   
9       Brazil         BRA           Exports of goods and services (% of GDP)   
10      Brazil         BRA  External balance on goods and services (% of GDP)   
11      Brazil         BRA  